# EA1 - Carga y creación de dataset: negocio comercial Olist

Flujo de la actividad:

```
HFDatasetLoader  →  DataFrames (tablas)  →  transformaciones en el notebook
                                         →  DatasetSummarizer  →  archivo .txt
```

- **Dataset:** [`bulutttt/olist-raw-data`](https://huggingface.co/datasets/bulutttt/olist-raw-data),
  copia pública del *Brazilian E-Commerce Public Dataset by Olist*. Son datos reales
  y anonimizados de un marketplace comercial brasileño (2016-2018).
- **Modelo de negocio:** clientes que compran productos a vendedores (proveedores)
  mediante órdenes, que se facturan con uno o varios pagos.

| Tabla | Archivo | Rol en el negocio |
|---|---|---|
| `clientes` | `olist_customers_dataset.csv` | Clientes y su ubicación |
| `ordenes` | `olist_orders_dataset.csv` | Encabezado de la orden/factura: estado y fechas |
| `items` | `olist_order_items_dataset.csv` | Detalle de la factura: producto, vendedor, precio y flete |
| `pagos` | `olist_order_payments_dataset.csv` | Pagos de cada orden: medio de pago y cuotas |
| `productos` | `olist_products_dataset.csv` | Catálogo de productos |
| `vendedores` | `olist_sellers_dataset.csv` | Vendedores/proveedores del marketplace |
| `categorias` | `product_category_name_translation.csv` | Traducción de categorías portugués → inglés |

- **Clase de carga:** `HFDatasetLoader` (`src/clases_po/hf_loader.py`) se conecta a Hugging Face y devuelve cada tabla como `pandas.DataFrame`.
- **Clase de resumen:** `DatasetSummarizer` (`src/clases_po/dataset_summarizer.py`) recibe un DataFrame y genera el resumen en `.txt`.

## 1. Preparar el entorno e importar las clases

Se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`, para importar las dos clases desde sus archivos `.py`.

In [1]:
import sys
from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

import pandas as pd
from datasets.utils.logging import disable_progress_bar, set_verbosity_error
from huggingface_hub.utils import logging as hf_logging

# Silenciar avisos y barras de progreso de la descarga
set_verbosity_error()
hf_logging.set_verbosity_error()
disable_progress_bar()

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import DatasetSummarizer, HFDatasetLoader

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 25)

## 2. Obtener los DataFrames desde Hugging Face

El repositorio publica cada tabla como un `.csv` independiente. `HFDatasetLoader` recibe el identificador del dataset, el archivo (`data_files`) y el `split`. Primero se carga una tabla con un loader individual y luego el resto con `load_tables`, que crea un loader por archivo y devuelve un diccionario `{tabla: DataFrame}`.

In [2]:
DATASET_ID = "bulutttt/olist-raw-data"

loader_ordenes = HFDatasetLoader(DATASET_ID, data_files="olist_orders_dataset.csv", split="train")
ordenes = loader_ordenes.load()
loader_ordenes.info()

{'dataset': 'bulutttt/olist-raw-data',
 'config': None,
 'data_files': 'olist_orders_dataset.csv',
 'split': 'train',
 'revision': None,
 'filas': 99441,
 'columnas': 8}

In [3]:
tablas = HFDatasetLoader.load_tables(
    DATASET_ID,
    {
        "clientes": "olist_customers_dataset.csv",
        "items": "olist_order_items_dataset.csv",
        "pagos": "olist_order_payments_dataset.csv",
        "productos": "olist_products_dataset.csv",
        "vendedores": "olist_sellers_dataset.csv",
        "categorias": "product_category_name_translation.csv",
    },
)
tablas["ordenes"] = ordenes

for nombre, df in tablas.items():
    print(f"{nombre:<11} {df.shape[0]:>7} filas x {df.shape[1]:>2} columnas | nulos: {int(df.isna().sum().sum())}")

clientes      99441 filas x  5 columnas | nulos: 0
items        112650 filas x  7 columnas | nulos: 0
pagos        103886 filas x  5 columnas | nulos: 0
productos     32951 filas x  9 columnas | nulos: 2448
vendedores     3095 filas x  4 columnas | nulos: 0
categorias       71 filas x  2 columnas | nulos: 0
ordenes       99441 filas x  8 columnas | nulos: 4908


In [4]:
clientes, items, pagos = tablas["clientes"], tablas["items"], tablas["pagos"]
productos, vendedores, categorias = tablas["productos"], tablas["vendedores"], tablas["categorias"]

items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [5]:
ordenes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## 3. Transformaciones

### Transformación 1: seleccionar, renombrar y traducir columnas

De `productos` solo se conservan el identificador, la categoría y el peso. La categoría viene en portugués, así que se une con la tabla de traducción y se renombran las columnas a español.

In [6]:
productos_t = (
    productos[["product_id", "product_category_name", "product_weight_g"]]
    .merge(categorias, on="product_category_name", how="left")
    .rename(columns={"product_category_name_english": "categoria", "product_weight_g": "peso_g"})
    .drop(columns="product_category_name")
)

productos_t.head()

,product_id,peso_g,categoria
0,1e9e8ef04dbcff4541ed26657ea517e5,225.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,1000.0,art
2,96bd76ec8810374ed1b65e291975717f,154.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,371.0,baby
4,9dc1a7de274444849c219cff195d0b71,625.0,housewares


### Transformación 2: manejar valores nulos

Hay productos sin categoría (nulos en el origen o categorías sin traducción). En lugar de eliminarlos, se reemplaza el nulo por `sin_categoria` para no perder esas ventas.

In [7]:
print(f"Productos sin categoría antes: {productos_t['categoria'].isna().sum()}")
productos_t["categoria"] = productos_t["categoria"].fillna("sin_categoria")
print(f"Productos sin categoría después: {productos_t['categoria'].isna().sum()}")

Productos sin categoría antes: 623
Productos sin categoría después: 0


### Transformación 3: convertir columnas de texto a fecha

Las fechas de `ordenes` llegan como texto (`object`). Se convierten a `datetime` para poder calcular tiempos y agrupar por mes.

In [8]:
columnas_fecha = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
ordenes_t = ordenes.copy()
ordenes_t[columnas_fecha] = ordenes_t[columnas_fecha].apply(pd.to_datetime, errors="coerce")

ordenes_t.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

### Transformación 4: filtrar filas

Para analizar ventas efectivas solo se conservan las órdenes con estado `delivered` que tienen fecha real de entrega.

In [9]:
print(ordenes_t["order_status"].value_counts(), end="\n\n")

ordenes_entregadas = ordenes_t[
    (ordenes_t["order_status"] == "delivered")
    & ordenes_t["order_delivered_customer_date"].notna()
]

print(f"Órdenes antes: {len(ordenes_t)} | entregadas con fecha: {len(ordenes_entregadas)}")

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Órdenes antes: 99441 | entregadas con fecha: 96470


### Transformación 5: agregar los pagos por orden (facturación)

Una orden puede pagarse en varias partes. Se agrupa `pagos` por `order_id` para obtener el total facturado, el número de cuotas y el medio de pago principal.

In [10]:
pagos_por_orden = (
    pagos.sort_values("payment_value", ascending=False)
    .groupby("order_id", as_index=False)
    .agg(
        total_pagado=("payment_value", "sum"),
        cuotas=("payment_installments", "max"),
        medio_pago=("payment_type", "first"),
    )
)

pagos_por_orden.head()

,order_id,total_pagado,cuotas,medio_pago
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


### Transformación 6: unir las tablas (`merge`)

Se combinan el detalle de la factura (`items`) con las órdenes entregadas, los productos, los vendedores (proveedores), los clientes y la facturación. El resultado es un DataFrame con una fila por producto vendido.

In [11]:
ventas = (
    items
    .merge(ordenes_entregadas, on="order_id", how="inner")
    .merge(productos_t, on="product_id", how="left")
    .merge(vendedores[["seller_id", "seller_state"]], on="seller_id", how="left")
    .merge(clientes[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
    .merge(pagos_por_orden, on="order_id", how="left")
    .rename(columns={
        "order_item_id": "linea",
        "price": "precio",
        "freight_value": "flete",
        "seller_state": "estado_vendedor",
        "customer_state": "estado_cliente",
        "order_purchase_timestamp": "fecha_compra",
        "order_delivered_customer_date": "fecha_entrega",
        "order_estimated_delivery_date": "fecha_estimada",
    })
)

print(f"items: {len(items)} filas -> ventas: {len(ventas)} filas")
ventas.head()

items: 112650 filas -> ventas: 110189 filas


,order_id,linea,product_id,seller_id,shipping_limit_date,precio,flete,customer_id,order_status,fecha_compra,order_approved_at,order_delivered_carrier_date,fecha_entrega,fecha_estimada,peso_g,categoria,estado_vendedor,customer_unique_id,estado_cliente,total_pagado,cuotas,medio_pago
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,650.0,cool_stuff,SP,871766c5855e863f6eccc05f988b23cb,RJ,72.19,2.0,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,30000.0,pet_shop,SP,eb28e67c4c0b83846050ddfb8a35d051,SP,259.83,3.0,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3050.0,furniture_decor,MG,3818d81c6709e39d06b2738a8d3a2474,MG,216.87,5.0,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,200.0,perfumery,SP,af861d436cfc08b2c2ddefd0ba074622,SP,25.78,2.0,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,3750.0,garden_tools,PR,64b576fb70d441e8f1b2d7d446e483c5,SP,218.04,3.0,credit_card


### Transformación 7: crear columnas calculadas

Se calculan el valor total de cada línea (precio + flete), los días de entrega, si la entrega llegó tarde, el mes de compra y si el vendedor y el cliente están en el mismo estado.

In [12]:
ventas["total_linea"] = ventas["precio"] + ventas["flete"]
ventas["dias_entrega"] = (ventas["fecha_entrega"] - ventas["fecha_compra"]).dt.days
ventas["entrega_tardia"] = ventas["fecha_entrega"].dt.normalize() > ventas["fecha_estimada"]
ventas["mes"] = ventas["fecha_compra"].dt.to_period("M").astype(str)
ventas["mismo_estado"] = ventas["estado_vendedor"] == ventas["estado_cliente"]

ventas[["order_id", "precio", "flete", "total_linea", "dias_entrega", "entrega_tardia", "mes", "mismo_estado"]].head()

,order_id,precio,flete,total_linea,dias_entrega,entrega_tardia,mes,mismo_estado
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19,7,False,2017-09,False
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83,16,False,2017-04,True
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87,7,False,2018-01,True
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78,6,False,2018-08,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04,25,False,2017-02,False


### Selección final de columnas

Se dejan solo las columnas útiles para el análisis, en un orden legible. Este es el DataFrame transformado que se resume en el `.txt`.

In [13]:
columnas_finales = [
    "order_id", "linea", "customer_unique_id", "seller_id", "product_id",
    "fecha_compra", "fecha_entrega", "fecha_estimada", "mes",
    "estado_cliente", "estado_vendedor", "mismo_estado", "categoria", "peso_g",
    "precio", "flete", "total_linea", "medio_pago", "cuotas", "total_pagado",
    "dias_entrega", "entrega_tardia",
]
ventas = ventas[columnas_finales].reset_index(drop=True)

print(ventas.shape)
ventas.head()

(110189, 22)


,order_id,linea,customer_unique_id,seller_id,product_id,fecha_compra,fecha_entrega,fecha_estimada,mes,estado_cliente,estado_vendedor,mismo_estado,categoria,peso_g,precio,flete,total_linea,medio_pago,cuotas,total_pagado,dias_entrega,entrega_tardia
0,00010242fe8c5a6d1ba2dd792cb16214,1,871766c5855e863f6eccc05f988b23cb,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,2017-09,RJ,SP,False,cool_stuff,650.0,58.90,13.29,72.19,credit_card,2.0,72.19,7,False
1,00018f77f2f0320c557190d7a144bdd3,1,eb28e67c4c0b83846050ddfb8a35d051,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,2017-04,SP,SP,True,pet_shop,30000.0,239.90,19.93,259.83,credit_card,3.0,259.83,16,False
2,000229ec398224ef6ca0657da4fc703e,1,3818d81c6709e39d06b2738a8d3a2474,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,2018-01,MG,MG,True,furniture_decor,3050.0,199.00,17.87,216.87,credit_card,5.0,216.87,7,False
3,00024acbcdf0a6daa1e931b038114c75,1,af861d436cfc08b2c2ddefd0ba074622,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,2018-08,SP,SP,True,perfumery,200.0,12.99,12.79,25.78,credit_card,2.0,25.78,6,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,64b576fb70d441e8f1b2d7d446e483c5,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,2017-02,SP,PR,False,garden_tools,3750.0,199.90,18.14,218.04,credit_card,3.0,218.04,25,False


### Transformación 8: agrupar por proveedor y por categoría/mes

Con el DataFrame transformado se construyen dos tablas agregadas: el desempeño de cada vendedor (proveedor) y las ventas por categoría y mes. Se guardan en `datos/` como evidencia.

In [14]:
ventas_por_vendedor = (
    ventas.groupby(["seller_id", "estado_vendedor"], as_index=False)
    .agg(
        ordenes=("order_id", "nunique"),
        productos_vendidos=("linea", "count"),
        ingreso=("precio", "sum"),
        dias_entrega_prom=("dias_entrega", "mean"),
        pct_entrega_tardia=("entrega_tardia", "mean"),
    )
    .sort_values("ingreso", ascending=False)
    .reset_index(drop=True)
    .round(2)
)

ventas_por_categoria_mes = (
    ventas.groupby(["categoria", "mes"], as_index=False)
    .agg(ordenes=("order_id", "nunique"), ingreso=("precio", "sum"), flete=("flete", "sum"))
    .sort_values(["categoria", "mes"])
    .reset_index(drop=True)
    .round(2)
)

Path("datos").mkdir(exist_ok=True)
ventas_por_vendedor.to_csv("datos/ventas_por_vendedor.csv", index=False)
ventas_por_categoria_mes.to_csv("datos/ventas_por_categoria_mes.csv", index=False)

ventas_por_vendedor.head(10)

,seller_id,estado_vendedor,ordenes,productos_vendidos,ingreso,dias_entrega_prom,pct_entrega_tardia
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1124,1148,226987.93,14.55,0.11
1,53243585a1d6dc2643021fd1853d8905,BA,348,400,217940.44,12.94,0.03
2,4a3ca9315b744ce9f8e9374361493884,SP,1772,1949,196882.12,13.94,0.10
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,578,579,190917.14,12.84,0.09
4,7c67e1448b00f6e969d365cea6b010ab,SP,973,1355,186570.05,21.95,0.09
5,7e93a43ef30c4f03f38b393420bc753a,SP,319,322,165981.49,10.83,0.05
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1311,1548,159816.87,10.70,0.06
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1145,1155,139658.69,10.69,0.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,910,1420,138208.56,11.57,0.07
9,955fee9216a65b617aa5c0531780ce60,SP,1261,1472,131836.71,10.30,0.06


In [15]:
(
    ventas.groupby("categoria")
    .agg(ordenes=("order_id", "nunique"), ingreso=("precio", "sum"))
    .sort_values("ingreso", ascending=False)
    .head(10)
    .round(2)
)

,ordenes,ingreso
categoria,,
health_beauty,8647,1233131.72
watches_gifts,5493,1165898.98
bed_bath_table,9272,1023434.76
sports_leisure,7529,954673.55
computers_accessories,6529,888613.62
furniture_decor,6307,711927.69
housewares,5743,615628.69
cool_stuff,3559,610204.10
auto,3809,578849.35


## 4. Generar el resumen con `DatasetSummarizer`

El DataFrame **transformado** (`ventas`) se pasa a la clase de resumen, que genera `reportes/ventas_olist.txt`. También se resume la tabla agregada por vendedor.

In [16]:
metadatos = {"Dataset": DATASET_ID, "Split": "train", "Contenido": "Ventas entregadas (DataFrame transformado)"}
resumen = DatasetSummarizer(ventas, titulo="Ventas entregadas - Olist (DataFrame transformado)", metadatos=metadatos)
ruta = resumen.guardar("reportes/ventas_olist.txt")

print(f"Resumen guardado en: {ruta}\n")
resumen.imprimir()

Resumen guardado en: reportes/ventas_olist.txt



VENTAS ENTREGADAS - OLIST (DATAFRAME TRANSFORMADO)
Generado: 2026-10-25 19:44:10

Origen de los datos
-------------------
- Dataset: bulutttt/olist-raw-data
- Split: train
- Contenido: Ventas entregadas (DataFrame transformado)

Dimensiones
-----------
Filas: 110189
Columnas: 22
Filas duplicadas: 0
Memoria aproximada: 74,553.1 KB

Columnas y tipos
----------------
- order_id: object (96470 valores únicos)
- linea: int64 (21 valores únicos)
- customer_unique_id: object (93350 valores únicos)
- seller_id: object (2970 valores únicos)
- product_id: object (32214 valores únicos)
- fecha_compra: datetime64[ns] (95948 valores únicos)
- fecha_entrega: datetime64[ns] (95658 valores únicos)
- fecha_estimada: datetime64[ns] (445 valores únicos)
- mes: object (23 valores únicos)
- estado_cliente: object (27 valores únicos)
- estado_vendedor: object (22 valores únicos)
- mismo_estado: bool (2 valores únicos)
- categoria: object (72 valores únicos)
- peso_g: float64 (2171 valores únicos)
- precio: 

In [17]:
resumen_vendedores = DatasetSummarizer(
    ventas_por_vendedor,
    titulo="Ventas por vendedor (proveedor) - Olist",
    metadatos={**metadatos, "Contenido": "Ventas agrupadas por vendedor"},
)
print(f"Resumen guardado en: {resumen_vendedores.guardar('reportes/ventas_por_vendedor.txt')}")

Resumen guardado en: reportes/ventas_por_vendedor.txt


## 5. Conclusión: transformaciones realizadas

| # | Transformación | Operación de pandas | Resultado |
|---|---|---|---|
| 1 | Seleccionar, renombrar y traducir columnas de productos | `df[[...]]`, `merge`, `rename`, `drop` | `productos_t` con categoría en inglés y nombres en español |
| 2 | Manejar valores nulos | `fillna` | Productos sin categoría quedan como `sin_categoria` |
| 3 | Convertir texto a fecha | `pd.to_datetime` | Columnas de fecha con tipo `datetime64` |
| 4 | Filtrar filas | filtro booleano | Solo órdenes `delivered` con fecha de entrega |
| 5 | Agregar pagos por orden | `groupby` + `agg` | Total facturado, cuotas y medio de pago por orden |
| 6 | Unir seis tablas | `merge` | Una fila por producto vendido con cliente, vendedor y pago |
| 7 | Columnas calculadas | operaciones vectorizadas y `.dt` | `total_linea`, `dias_entrega`, `entrega_tardia`, `mes`, `mismo_estado` |
| 8 | Agrupar por vendedor y por categoría/mes | `groupby` + `agg` | `ventas_por_vendedor` y `ventas_por_categoria_mes` |

El archivo `reportes/ventas_olist.txt` corresponde al DataFrame transformado `ventas`: sus filas, columnas, tipos y nulos son los de los datos ya transformados y no los de las tablas originales.